# Dash Components

## Objectives
- Know how to add multiple components in a dashboard
- Handle multiple outputs from callbacks

## Dataset Used
[Airline Reporting Carrier On-Time Performance dataset](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv)

## Theme
Analyze flight delays in a dashboard

## Dashboard Components
- Monthly average carrier delay by reporting airline for the given year.
- Monthly average weather delay by reporting airline for the given year.
- Monthly average national air system delay by reporting airline for the given year.
- Monthly average security delay by reporting airline for the given year.
- Monthly average late aircraft delay by reporting airline for the given year.

Note: Year range is between 2010 and 2020

## Expected Output
The output should have three components:
- Application title
- Input box to enter year
- 5 Charts conveying different types of delay
    - Carrier and Weather delay in the first segment
    - National air system and security delay in the second
    - Late aircraft delay in the third

![expected-output](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/images/lab3_expected_output.png)

---

In [74]:
# Import required libraries
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, callback, Input, Output
import dash_bootstrap_components as dbc

# Import dataset
url = '/'.join(['https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud',
                'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv'])
data = pd.read_csv(url)
data.set_index('Year', inplace=True)
data.head()

,Unnamed: 0,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
Year,,,,,,,,,,,,,,,,,,,,,
1998,1295781,2,4,2,4,1998-04-02,AS,19930,AS,N785AS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013,1125375,2,5,13,1,2013-05-13,EV,20366,EV,N24103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993,118824,3,9,25,6,1993-09-25,UA,19977,UA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1994,634825,4,11,12,6,1994-11-12,HP,19991,HP,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017,1888125,3,8,17,4,2017-08-17,UA,19977,UA,N827UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
columns=['Month','Reporting_Airline','CarrierDelay','WeatherDelay',
         'NASDelay','SecurityDelay','LateAircraftDelay']
grouped_data = data.loc[2011,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
grouped_data.head()
# Example - CarrierDelay for 2011
#px.line(grouped_data,x='Month',y='CarrierDelay',line_group='Reporting_Airline',color='Reporting_Airline')

,Reporting_Airline,Month,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,AA,1,NaN,NaN,NaN,NaN,NaN
1,AA,2,10.0,0.0,0.000000,0.0,22.0
2,AA,3,0.0,0.0,18.666667,0.0,19.0
3,AA,4,31.5,0.0,0.000000,0.0,52.0
4,AA,5,0.0,0.0,0.000000,0.0,83.0


In [110]:
#Initialize App - incorporate a Dash Bootstrap theme
external_stylesheets = [dbc.themes.CERULEAN]
app = Dash(__name__, external_stylesheets=external_stylesheets)

#App Layout
app.layout = dbc.Container([
    html.H1(
        'Flight Delay Time Statistics',
        style={'color': '#503D36','font-size': 40,'textAlign': 'right'}
    ),
    
    dbc.Row([
        dbc.Col(['Input Year: ']),
        dbc.Col([
            dcc.Input(
                id = 'input-year',
                value = 2010,
                type = 'number',
                style = {'height': '50px', 'font-size': 35}
            )
        ])
    ], style={'font-size': 40}),

    dbc.Row([
        dbc.Col(
            [dcc.Graph(figure = {}, id='carrier-delay')],
            #width = 6
        ),
        dbc.Col(
            [dcc.Graph(figure = {}, id='weather-delay')],
            #width = 6
        ),
    ]),

    dbc.Row([
        dbc.Col(
            [dcc.Graph(figure = {}, id='nas-delay')],
            #width = 6
        ),
        dbc.Col(
            [dcc.Graph(figure = {}, id='security-delay')],
            #width = 6
        ),
    ]),

    dcc.Graph(figure={}, id='late-aircraft-delay')
])

# Callback
@callback(
    Output('carrier-delay','figure'),
    Input('input-year','value')
)
def carrier_delay_graph(year):
    columns = ['Reporting_Airline','Month','CarrierDelay']
    grouped_data = data.loc[year,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
    fig = px.line(
        grouped_data,
        x='Month',
        y='CarrierDelay',
        line_group='Reporting_Airline',
        color='Reporting_Airline',
        title='Average carrier delay time (minutes) by airline'
    )
    return fig

@callback(
    Output('weather-delay','figure'),
    Input('input-year','value')
)
def weather_delay_graph(year):
    columns = ['Reporting_Airline','Month','WeatherDelay']
    grouped_data = data.loc[year,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
    fig = px.line(
        grouped_data,
        x='Month',
        y='WeatherDelay',
        line_group='Reporting_Airline',
        color='Reporting_Airline',
        title='Average weather delay time (minutes) by airline'
    )
    return fig
    
@callback(
    Output('nas-delay','figure'),
    Input('input-year','value')
)
def nas_delay_graph(year):
    columns = ['Reporting_Airline','Month','NASDelay']
    grouped_data = data.loc[year,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
    fig = px.line(
        grouped_data,
        x='Month',
        y='NASDelay',
        line_group='Reporting_Airline',
        color='Reporting_Airline',
        title='Average NAS delay time (minutes) by airline'
    )
    return fig

@callback(
    Output('security-delay','figure'),
    Input('input-year','value')
)
def security_delay_graph(year):
    columns = ['Reporting_Airline','Month','SecurityDelay']
    grouped_data = data.loc[year,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
    fig = px.line(
        grouped_data,
        x='Month',
        y='SecurityDelay',
        line_group='Reporting_Airline',
        color='Reporting_Airline',
        title='Average security delay time (minutes) by airline'
    )
    return fig

@callback(
    Output('late-aircraft-delay','figure'),
    Input('input-year','value')
)
def late_aircraft_delay_graph(year):
    columns = ['Reporting_Airline','Month','LateAircraftDelay']
    grouped_data = data.loc[year,columns].groupby(['Reporting_Airline','Month'],as_index=False).mean()
    fig = px.line(
        grouped_data,
        x='Month',
        y='LateAircraftDelay',
        line_group='Reporting_Airline',
        color='Reporting_Airline',
        title='Average late aircraft delay time (minutes) by airline'
    )
    return fig

In [111]:
if __name__ == '__main__':
    app.run(jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/
